# System Vicarious Calibration

In [ ]:
image_l1 = aabim.Image.from_nc("aabim_image_l1.nc")
in_situ = aabim.InSitu.from_csv("in_situ_rho_w.csv")

cal_data = aabim.CalibrationData.compute(image_l1, in_situ)
cal_data.plot()
cal_data.summary()
cal_data.save("cal_data.nc")

cal_model = aabim.CalibrationModel.fit(cal_data, formula="rho_t_hat ~ rho_t * a + b")
cal_model.summary()
cal_model.plot()
cal_model.save("cal_model.pkl")

image_l1c = image_l1.calibrate(cal_model, output="image_l1c.nc")

# Classic atmospheric correction

In [ ]:
import aabim

image_l1 = aabim.Image.from_nc("aabim_image_l1.nc")
cal_model = aabim.CalibrationModel.load("cal_model.pkl")

image_l2 = image_l1.correct(cal_model, n_cores=10, output="image_l2.nc")

# Aquatic Inversion 

In [ ]:
import aabim

image_l2 = aabim.Image.from_nc("aabim_image_l2.nc")

r_b_prior = aabim.SpectralPrior.from_csv(
    "r_b_library.csv", q=7  # number of PCA components
)

r_b_prior_interp = r_b_prior.interpolate(image.wavelength)

# r_b_prior exposes what it computed
r_b_prior_interp.components.plot()
r_b_prior_interp.explained_variance.plot()
r_b_prior_interp.enveloppe.plot()

# create the model from image and prior knowledge
model = aabim.AquaticModel(
    a_nap_star=(0.0051, 0.00001),
    bb_p_star=(0.0047, 0.00001),
    a_g_s=(0.017, 0.001),
    a_nap_s=(0.006, 0.0001),
    bb_p_s=(0.65, 0.01),
    h_w=(5, 5),
    r_b_prior=r_b_prior_interp,
)

result = image_l2.invert(model, output="aabim_image_atmospheric_aquatic_inversion.nc")

# Atmospheric + Aquatic Inversion

In [ ]:
import aabim

image = aabim.Image.from_nc("aabim_image_l1c.nc")

r_b_prior = aabim.SpectralPrior.from_csv(
    "r_b_library.csv", q=7  # number of PCA components
)

r_b_prior_interp = r_b_prior.interpolate(image.wavelength)

# r_b_prior exposes what it computed
r_b_prior_interp.components.plot()
r_b_prior_interp.explained_variance.plot()
r_b_prior_interp.enveloppe.plot()

# create the model data from image and prior knowledge
model = aabim.AtmosphericAquaticModel(
    aod_550=(0.05, 0.05),
    a_nap_star=(0.0051, 0.00001),
    bb_p_star=(0.0047, 0.00001),
    a_g_s=(0.017, 0.001),
    a_nap_s=(0.006, 0.0001),
    bb_p_s=(0.65, 0.01),
    h_w=(5, 5),
    r_b_prior=r_b_prior_interp,
)

result = image_l1c.invert(model, output="aabim_image_atmospheric_aquatic_inversion.nc")